In [1]:
import networkx as nx 
import pandas as pd 
import numpy as np
import pickle
import os
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
# scipy dependency for PPR

from statsmodels.stats.multitest import multipletests

from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr
from rpy2.robjects.vectors import ListVector, StrVector

In [2]:
# Load all networks built from previous notebook 

def pickle_load(in_file):
    with open(in_file,'rb') as input_f:
        var = pickle.load(input_f)
    return var

string_nx  = pickle_load("../data/network/string_physical.pkl")

In [3]:
# Load seed genes/nodes from the cfg file
seed_nodes = pd.read_csv("../cfg/seed_nodes_CP_sym.txt", header=None).values.flatten()
print(seed_nodes)

['PTPRD' 'NTM' 'LAPTM4A' 'SDC1' 'KCNQ3' 'PTK7' 'CUL9']


In [4]:
# Load seed genes/nodes weights from the cfg file
seed_weights = pd.read_csv("../cfg/seed_nodes_CP_weights.txt", sep=",")
(seed_weights)

# tot = Sum(Weight) --> indicates number of lead snps
# PPR Prob is weight_i/sum(weight) 
# Weight = 1 --> PPR prob = 1/sum(weight) (1/nLeads)
# Weight = .5 --> PPR prob = .5/sum(weight) (1/nLeads*2)

# Total sum must be 1

,Gene,Weight
0,PTPRD,1.0
1,NTM,1.0
2,LAPTM4A,0.5
3,SDC1,0.5
4,KCNQ3,1.0
5,PTK7,0.5
6,CUL9,0.5


In [5]:
seed_weights['Weight'].sum()

np.float64(5.0)

In [6]:
seed_weights[seed_weights['Gene'] == 'PTPRD']['Weight']

0    1.0
Name: Weight, dtype: float64

In [7]:
def ppr_network(seed_nodes, network, name):
    common_nodes = set(seed_nodes) & set(network.nodes)
    print(f"{len(common_nodes)} nodes in common, i.e.: {common_nodes}")

    # build personalized
    personalization = {
        seed: (seed_weights[seed_weights['Gene'] == seed]['Weight']/seed_weights['Weight'].sum()).values[0] for seed in common_nodes
    }

    # CHOSEN alternative --> uniform
    personalization = {
        seed: 1/len(common_nodes) for seed in common_nodes
    }

    print(personalization)
    assert len(set(network.nodes)) == len(list(network.nodes))

    ppr = nx.pagerank(
        G=network,
        personalization=personalization,
    )

    mean_ = np.mean(list(ppr.values()))
    std_ = np.std(list(ppr.values()))

    ppr_df = pd.DataFrame.from_dict(ppr, orient='index', columns=['ppr_importance'])
    ppr_df['zscore'] = (ppr_df['ppr_importance'] - mean_)/std_
    ppr_df = ppr_df.sort_values(['ppr_importance'], ascending=False)
    ppr_df['rank'] = list(range(1,len(ppr_df)+1))
    ppr_df = ppr_df.add_suffix('_'+name)
    return ppr_df
        

In [8]:
networks = {
    'string':   string_nx,
}

ppr_results = {}

for k,v in tqdm(networks.items()):
    print(v)
    ppr_results[k] =  ppr_network(seed_nodes, v, k)


  0%|          | 0/1 [00:00<?, ?it/s]

Graph with 14280 nodes and 143496 edges
7 nodes in common, i.e.: {'LAPTM4A', 'NTM', 'KCNQ3', 'CUL9', 'SDC1', 'PTK7', 'PTPRD'}
{'LAPTM4A': 0.14285714285714285, 'NTM': 0.14285714285714285, 'KCNQ3': 0.14285714285714285, 'CUL9': 0.14285714285714285, 'SDC1': 0.14285714285714285, 'PTK7': 0.14285714285714285, 'PTPRD': 0.14285714285714285}


In [9]:
merged_df = pd.DataFrame()

for k,df in ppr_results.items():
    merged_df = (
        merged_df.merge(
            df,
            how='outer',
            right_index=True,
            left_index=True,
            validate='one_to_one'
        )
    )

In [10]:
# i want genes that are in at least 3 networks
# threshold controls the minimum number of networks
merged_df['Gene'] = merged_df.index 
merged_df.to_parquet("../output/ppr_results.parquet", compression="zstd")

In [11]:
merged_df.sort_values('rank_string')

,ppr_importance_string,zscore_string,rank_string,Gene
PTPRD,3.183950e-02,49.162583,1,PTPRD
NTM,2.900224e-02,44.771989,2,NTM
SDC1,2.580097e-02,39.818093,3,SDC1
LAPTM4A,2.412639e-02,37.226708,4,LAPTM4A
KCNQ3,2.408767e-02,37.166788,5,KCNQ3
...,...,...,...,...
ZNF331,3.965803e-07,-0.107753,14276,ZNF331
FGGY,3.849975e-07,-0.107771,14277,FGGY
KIAA0825,3.675146e-07,-0.107798,14278,KIAA0825
MLIP,3.675146e-07,-0.107798,14279,MLIP


In [12]:
merged_df.sort_values('rank_string').to_csv("../output/full_ppr.csv")

top_genes = merged_df.sort_values('rank_string').iloc[:int(len(merged_df)*0.01)]
top_genes.to_csv("../output/top_1perc_genes.csv")